In [3]:
# %%
# Channel subset models — lateral rows + BIS Quatro approximation
# 4 subsets × 3 feature sets (A/B/C) × 5 folds = 60 outer tasks
# 60 tasks mapped to 60 cores: each outer task runs GridSearchCV(n_jobs=1)
from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)
from sklearn.model_selection import GroupKFold, GridSearchCV

import mne
mne.set_log_level("WARNING")

In [5]:
# %%
PROJECT_ROOT  = Path("..").resolve()
DERIVED_ROOT  = PROJECT_ROOT / "data" / "derived"
MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEAT_A_PATH   = DERIVED_ROOT / "features" / "features_A_bandpower_epochwise.csv"
MATRICES_PATH = DERIVED_ROOT / "features" / "features_B_wpli_matrices_epoch_sliding.npz"
OUT_DIR       = PROJECT_ROOT / "results" / "models"
FIGURE_DIR    = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ── Channel subsets ──────────────────────────────────────────────────────────
# T3=T7, T4=T8, T5=P7, T6=P8 in modern 10-20 notation
SUBSETS = {
    "left_lateral":  ["Fp1", "F7", "T7", "P7", "O1"],
    "right_lateral": ["Fp2", "F8", "T8", "P8", "O2"],
    "both_lateral":  ["Fp1", "F7", "T7", "P7", "O1",
                      "Fp2", "F8", "T8", "P8", "O2"],
    # BIS Quatro (4-electrode strip, left-sided placement):
    #   E1 = Fpz  (center forehead)
    #   E2 = Fp1  (just lateral to centre)
    #   E3 = F7   (temple, between eye corner and hairline)
    #   E4 = AF3  (above eyebrow, prefrontal — closest available to Fp1/2 above brow)
    "bis_quatro":    ["Fpz", "Fp1", "AF3", "F7"],
}

BANDS_BP   = {"delta": (1.0, 4.0), "theta": (4.0, 8.0),
              "alpha": (8.0, 13.0), "beta": (13.0, 30.0)}
BANDS_WPLI = ["theta", "alpha", "beta"]
EYES_KEEP  = "closed"
REJECT_PTP_UV = 250.0

OUTER_SPLITS = 5
INNER_SPLITS = 4
SEED         = 0
N_CORES      = os.cpu_count() or 8

N_MODELS     = len(SUBSETS) * 3          # 4 subsets × A/B/C
N_TASKS      = N_MODELS * OUTER_SPLITS   # 60
N_OUTER_JOBS = min(N_CORES, N_TASKS)     # 60
N_INNER_JOBS = max(1, N_CORES // N_OUTER_JOBS)  # 1

PARAM_GRID = {
    "n_estimators":      [200, 400, 800],
    "max_depth":         [5, 10, 20],
    "min_samples_split": [2, 6, 10],
    "min_samples_leaf":  [2, 4, 5],
    "max_features":      ["sqrt", 0.3, 0.8],
}
n_combos = 1
for v in PARAM_GRID.values():
    n_combos *= len(v)

print(f"Cores: {N_CORES}  |  Outer tasks: {N_TASKS}  |  Outer jobs: {N_OUTER_JOBS}  |  Inner jobs: {N_INNER_JOBS}")
print(f"Grid: {n_combos} combos × {INNER_SPLITS} inner folds = {n_combos*INNER_SPLITS:,} fits per task")
print(f"Total fits: {n_combos * INNER_SPLITS * N_TASKS:,}")
print()
for name, chs in SUBSETS.items():
    n_e = len(chs)*(len(chs)-1)//2
    print(f"  {name}: {len(chs)} ch | {len(chs)*4} bp feats | {n_e*3} wPLI feats | {len(chs)*4 + n_e*3} total")

Cores: 60  |  Outer tasks: 60  |  Outer jobs: 60  |  Inner jobs: 1
Grid: 243 combos × 4 inner folds = 972 fits per task
Total fits: 58,320

  left_lateral: 5 ch | 20 bp feats | 30 wPLI feats | 50 total
  right_lateral: 5 ch | 20 bp feats | 30 wPLI feats | 50 total
  both_lateral: 10 ch | 40 bp feats | 135 wPLI feats | 175 total
  bis_quatro: 4 ch | 16 bp feats | 18 wPLI feats | 34 total


In [7]:
# %%
# ============================================
# Section 1. Channel metadata + index maps
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)
sample_ep = mne.io.read_epochs_eeglab(manifest["file_path"].iloc[0], verbose="ERROR")
CH_NAMES  = sample_ep.ch_names
CH_INFO   = sample_ep.info

# Validate all channels exist
all_needed = sorted({c for chs in SUBSETS.values() for c in chs})
missing = [c for c in all_needed if c not in CH_NAMES]
assert not missing, f"Missing channels: {missing}"
print(f"All {len(all_needed)} needed channels found: {all_needed}")

@dataclass
class SubsetMeta:
    name:       str
    channels:   list[str]
    ch_idx:     list[int]       = field(init=False)
    triu_r:     np.ndarray      = field(init=False)
    triu_c:     np.ndarray      = field(init=False)
    bp_cols:    list[str]       = field(init=False)
    wpli_cols:  list[str]       = field(init=False)

    def __post_init__(self):
        self.ch_idx = [CH_NAMES.index(c) for c in self.channels]
        n = len(self.channels)
        self.triu_r, self.triu_c = np.triu_indices(n, k=1)
        self.bp_cols = [
            f"bp_{band}_{ch}"
            for band in BANDS_BP
            for ch in self.channels
        ]
        self.wpli_cols = [
            f"{band}_{self.channels[r]}-{self.channels[c]}"
            for band in BANDS_WPLI
            for r, c in zip(self.triu_r, self.triu_c)
        ]

SUBSET_META = {name: SubsetMeta(name, chs) for name, chs in SUBSETS.items()}

for sm in SUBSET_META.values():
    print(f"{sm.name}: idx={sm.ch_idx}  bp={len(sm.bp_cols)}  wpli={len(sm.wpli_cols)}")

All 12 needed channels found: ['AF3', 'F7', 'F8', 'Fp1', 'Fp2', 'Fpz', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']
left_lateral: idx=[61, 55, 37, 17, 3]  bp=20  wpli=30
right_lateral: idx=[59, 47, 29, 9, 1]  bp=20  wpli=30
both_lateral: idx=[61, 55, 37, 17, 3, 59, 47, 29, 9, 1]  bp=40  wpli=135
bis_quatro: idx=[60, 61, 58, 55]  bp=16  wpli=18


In [8]:
# %%
# ============================================
# Section 2. Extract bandpower for ALL needed channels in one pass
# ============================================

all_ch_set = sorted({c for chs in SUBSETS.values() for c in chs})
df_manifest_ec = manifest[manifest["eyes"] == EYES_KEEP].sort_values(
    ["subject_id", "recording_number"]
).reset_index(drop=True)

bp_rows = []
for _, row in df_manifest_ec.iterrows():
    epochs = mne.io.read_epochs_eeglab(row["file_path"], verbose="ERROR")
    epochs.load_data()
    epochs.pick(all_ch_set)   # load only the 12 channels we need

    local_ch = epochs.ch_names   # order after pick()
    data  = epochs.get_data()    # (n_ep, n_ch, n_times)
    sfreq = float(epochs.info["sfreq"])

    ptp_uv = np.ptp(data, axis=-1).max(axis=1) * 1e6
    keep   = np.where(ptp_uv <= REJECT_PTP_UV)[0]

    psds, freqs = mne.time_frequency.psd_array_welch(
        data[keep], sfreq=sfreq, fmin=1.0, fmax=40.0,
        n_fft=data.shape[-1], n_overlap=0, verbose="ERROR",
    )  # (n_kept, n_ch, n_freqs)

    for ep_pos, orig_idx in enumerate(keep):
        feat = {}
        for band_name, (flo, fhi) in BANDS_BP.items():
            mask = (freqs >= flo) & (freqs < fhi)
            for ch_name in all_ch_set:
                ci = local_ch.index(ch_name)
                feat[f"bp_{band_name}_{ch_name}"] = float(
                    np.log(np.maximum(psds[ep_pos, ci, mask].mean(), 1e-20))
                )
        bp_rows.append({
            "subject_id":         row["subject_id"],
            "recording_number":   int(row["recording_number"]),
            "epoch_index_original": int(orig_idx),
            "drug":               row["drug"],
            **feat,
        })

bp_df = pd.DataFrame(bp_rows)
print(f"Bandpower rows: {len(bp_df)}  |  channels extracted: {len(all_ch_set)}")

Bandpower rows: 276  |  channels extracted: 12


In [10]:
# %%
# ============================================
# Section 3. Build per-subset feature matrices from NPZ matrices
# ============================================

npz = np.load(MATRICES_PATH)

# For each subset, build (n_epochs, n_wpli_features) from the 62×62 matrices
wpli_arrays: dict[str, np.ndarray] = {}

for sm in SUBSET_META.values():
    rows = []
    for _, row in bp_df.iterrows():
        sid    = str(int(row["subject_id"]))
        recnum = int(row["recording_number"])
        eidx   = int(row["epoch_index_original"])
        edge_vec = []
        for band in BANDS_WPLI:
            mat = npz[f"{sid}__rec{recnum}__e{eidx:04d}__{band}"]
            sub = mat[np.ix_(sm.ch_idx, sm.ch_idx)]
            edge_vec.append(sub[sm.triu_r, sm.triu_c])
        rows.append(np.concatenate(edge_vec))
    wpli_arrays[sm.name] = np.array(rows)
    print(f"{sm.name} wPLI: {wpli_arrays[sm.name].shape}")

left_lateral wPLI: (276, 30)
right_lateral wPLI: (276, 30)
both_lateral wPLI: (276, 135)
bis_quatro wPLI: (276, 18)


In [11]:
# %%
# ============================================
# Section 4. Assemble all (subset, model) specs
# ============================================

y      = (bp_df["drug"] == "ketamine").astype(int).to_numpy()
groups = bp_df["subject_id"].astype(str).to_numpy()

@dataclass(frozen=True)
class ModelSpec:
    X:          np.ndarray
    model_name: str
    subset:     str
    feat_type:  str   # A / B / C
    n_features: int

all_specs: list[ModelSpec] = []
for sm in SUBSET_META.values():
    XA = bp_df[sm.bp_cols].to_numpy()
    XB = wpli_arrays[sm.name]
    XC = np.concatenate([XA, XB], axis=1)
    for feat_type, X in [("A", XA), ("B", XB), ("C", XC)]:
        all_specs.append(ModelSpec(
            X=X,
            model_name=f"{sm.name}__{feat_type}",
            subset=sm.name,
            feat_type=feat_type,
            n_features=X.shape[1],
        ))

print(f"Total model specs: {len(all_specs)}")
for s in all_specs:
    print(f"  {s.model_name}: {s.X.shape}")

Total model specs: 12
  left_lateral__A: (276, 20)
  left_lateral__B: (276, 30)
  left_lateral__C: (276, 50)
  right_lateral__A: (276, 20)
  right_lateral__B: (276, 30)
  right_lateral__C: (276, 50)
  both_lateral__A: (276, 40)
  both_lateral__B: (276, 135)
  both_lateral__C: (276, 175)
  bis_quatro__A: (276, 16)
  bis_quatro__B: (276, 18)
  bis_quatro__C: (276, 34)


In [9]:
# %%
# ============================================
# Section 5. Nested CV with exhaustive GridSearchCV
# 60 outer tasks × 1 inner job = 60 cores used simultaneously
# ============================================

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
splits   = list(outer_cv.split(all_specs[0].X, y, groups))


def fit_eval_one(X: np.ndarray, model_name: str, subset: str, feat_type: str,
                 fold: int, tr: np.ndarray, te: np.ndarray) -> tuple:
    inner_cv = GroupKFold(n_splits=INNER_SPLITS)
    gtr, gte = groups[tr], groups[te]

    gs = GridSearchCV(
        RandomForestClassifier(
            random_state=SEED, class_weight="balanced_subsample", n_jobs=1
        ),
        param_grid=PARAM_GRID,
        cv=inner_cv,
        scoring="balanced_accuracy",
        n_jobs=N_INNER_JOBS,
        refit=True,
    )
    gs.fit(X[tr], y[tr], groups=gtr)
    best  = gs.best_estimator_
    proba = best.predict_proba(X[te])[:, 1]
    yhat  = (proba >= 0.5).astype(int)

    try:
        auc = float(roc_auc_score(y[te], proba))
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y[te], yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (np.nan,) * 4

    fold_row = {
        "model": model_name, "subset": subset, "feat_type": feat_type, "fold": fold,
        "n_test": int(len(te)), "n_test_subjects": int(len(np.unique(gte))),
        "accuracy":          float(accuracy_score(y[te], yhat)),
        "balanced_accuracy": float(balanced_accuracy_score(y[te], yhat)),
        "roc_auc":           auc,
        "best_score_inner":  float(gs.best_score_),
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }
    pred_rows = [
        {"model": model_name, "subset": subset, "feat_type": feat_type, "fold": fold,
         "subject_id": gte[i], "y_true": int(y[te][i]),
         "y_proba": float(proba[i]), "y_pred": int(yhat[i])}
        for i in range(len(te))
    ]
    param_row = {
        "model": model_name, "subset": subset, "feat_type": feat_type, "fold": fold,
        "best_score_inner": float(gs.best_score_),
        **gs.best_params_,
    }
    return fold_row, pred_rows, param_row


tasks = [
    (s.X, s.model_name, s.subset, s.feat_type, fold, tr, te)
    for s in all_specs
    for fold, (tr, te) in enumerate(splits, start=1)
]
print(f"Launching {len(tasks)} tasks across {N_OUTER_JOBS} workers …")

results = Parallel(n_jobs=N_OUTER_JOBS, prefer="processes")(
    delayed(fit_eval_one)(*t) for t in tasks
)

folds_df  = pd.DataFrame([r[0] for r in results]).sort_values(["subset", "feat_type", "fold"]).reset_index(drop=True)
preds_df  = pd.DataFrame([pr for r in results for pr in r[1]])
params_df = pd.DataFrame([r[2] for r in results]).sort_values(["subset", "feat_type", "fold"]).reset_index(drop=True)

print("Done.")
display(folds_df.groupby(["subset", "feat_type"])[["balanced_accuracy", "roc_auc"]].mean().round(4))

Launching 60 tasks across 60 workers …
Done.


balanced_accuracy  roc_auc
subset        feat_type                            
bis_quatro    A                     0.6333   0.7031
              B                     0.4625   0.4547
              C                     0.5995   0.6723
both_lateral  A                     0.7540   0.8182
              B                     0.4770   0.4380
              C                     0.7273   0.8019
left_lateral  A                     0.7310   0.8014
              B                     0.4979   0.5448
              C                     0.7353   0.8057
right_lateral A                     0.7043   0.7701
              B                     0.3584   0.3280
              C                     0.6436   0.7209

In [10]:
# %%
# Save
folds_df.to_csv(OUT_DIR / "results_rf_channel_subsets.csv", index=False)
preds_df.to_csv(OUT_DIR / "predictions_rf_channel_subsets.csv", index=False)
params_df.to_csv(OUT_DIR / "best_params_rf_channel_subsets.csv", index=False)
print("Saved.")

Saved.


In [12]:
# %%
# ============================================
# Section 5b. Permutation null distribution
# Subject-level shuffle of recording labels (GroupKFold safe)
# Refit per fold using saved best params (loaded from CSV — no GridSearchCV)
# Two-sided permutation p-value vs null mean
# ============================================

N_PERM = 1000

# Load saved fold results + best params from CSV (skips needing the GridSearchCV cell)
folds_df  = pd.read_csv(OUT_DIR / "results_rf_channel_subsets.csv")
params_df = pd.read_csv(OUT_DIR / "best_params_rf_channel_subsets.csv")

# Recording-level shuffle structures
rec_id_arr = (bp_df["subject_id"].astype(str) + "||" +
              bp_df["recording_number"].astype(str)).to_numpy()
rec_lookup = pd.DataFrame({"rec_id": rec_id_arr, "y": y, "subject": groups})
assert (rec_lookup.groupby("rec_id")["y"].nunique() <= 1).all(), \
    "Label not constant within recording."

rec_label   = rec_lookup.groupby("rec_id")["y"].first()
rec_subject = rec_lookup.groupby("rec_id")["subject"].first()
rec_ids_unique = rec_label.index.to_numpy()
rec_y          = rec_label.to_numpy().astype(int)
rec_subj       = rec_subject.to_numpy()

epoch_to_rec_idx = pd.Index(rec_ids_unique).get_indexer(rec_id_arr)
assert (epoch_to_rec_idx >= 0).all()

subj_to_rec_idxs = {}
for i, s in enumerate(rec_subj):
    subj_to_rec_idxs.setdefault(s, []).append(i)
subj_rec_idx_list = [np.asarray(v, dtype=int)
                     for v in subj_to_rec_idxs.values() if len(v) > 1]

# Sanitize best params (CSV roundtrip turns ints to floats and "sqrt" to str)
INT_KEYS = {"n_estimators", "max_depth", "min_samples_split", "min_samples_leaf"}

def _coerce(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if kind == "int":
        try:
            return int(float(v))
        except (TypeError, ValueError):
            return v
    if kind == "max_features":
        if isinstance(v, str):
            s = v.strip()
            if s in {"sqrt", "log2"}:
                return s
            try:
                f = float(s)
            except ValueError:
                return s
            return int(f) if f.is_integer() else f
        if isinstance(v, float) and float(v).is_integer():
            return int(v)
        return v
    return v

BEST_BY_MODEL_FOLD = {}
for _, prow in params_df.iterrows():
    rfp = {}
    for k in PARAM_GRID.keys():
        if k in prow:
            rfp[k] = _coerce(prow[k], "int" if k in INT_KEYS else
                                       "max_features" if k == "max_features" else None)
    BEST_BY_MODEL_FOLD[(prow["model"], int(prow["fold"]))] = rfp

def make_estimator(model_name, fold):
    rf = RandomForestClassifier(
        random_state=SEED, class_weight="balanced_subsample", n_jobs=1
    )
    rf.set_params(**BEST_BY_MODEL_FOLD[(model_name, fold)])
    return rf

# Splits are deterministic given y/groups — rebuild here so the cell is self-contained
outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
splits   = list(outer_cv.split(all_specs[0].X, y, groups))

# Permutation worker
def run_perm_chunk(seed_list):
    out_chunk = []
    for seed in seed_list:
        rng = np.random.default_rng(seed)
        perm_rec_y = rec_y.copy()
        for idxs in subj_rec_idx_list:
            vals = perm_rec_y[idxs].copy()
            rng.shuffle(vals)
            perm_rec_y[idxs] = vals
        y_perm = perm_rec_y[epoch_to_rec_idx]

        per_bacc, per_auc = {}, {}
        for spec in all_specs:
            fb, fa = [], []
            for fold, (tr, te) in enumerate(splits, start=1):
                est = make_estimator(spec.model_name, fold)
                est.fit(spec.X[tr], y_perm[tr])
                proba = est.predict_proba(spec.X[te])[:, 1]
                yhat  = (proba >= 0.5).astype(int)
                fb.append(balanced_accuracy_score(y_perm[te], yhat))
                try:
                    fa.append(roc_auc_score(y_perm[te], proba))
                except Exception:
                    fa.append(np.nan)
            per_bacc[spec.model_name] = float(np.nanmean(fb))
            per_auc[spec.model_name]  = float(np.nanmean(fa))
        out_chunk.append({"bacc": per_bacc, "auc": per_auc})
    return out_chunk

seeds = [SEED + 1000 + i for i in range(N_PERM)]
chunks = [c.tolist() for c in np.array_split(np.array(seeds, dtype=int), N_CORES)
          if len(c) > 0]

print(f"Running {N_PERM} permutations across {len(chunks)} workers …")
chunked = Parallel(n_jobs=N_CORES, prefer="processes", batch_size=1)(
    delayed(run_perm_chunk)(c) for c in chunks
)
perm_flat = [r for sub in chunked for r in sub]

null_bacc = {s.model_name: np.array([r["bacc"][s.model_name] for r in perm_flat])
             for s in all_specs}
null_auc  = {s.model_name: np.array([r["auc"][s.model_name]  for r in perm_flat])
             for s in all_specs}

# Two-sided permutation p-value vs null mean
obs_bacc = folds_df.groupby(["subset", "feat_type"])["balanced_accuracy"].mean()
obs_auc  = folds_df.groupby(["subset", "feat_type"])["roc_auc"].mean()

perm_rows = []
for s in all_specs:
    ob = float(obs_bacc.loc[s.subset, s.feat_type])
    oa = float(obs_auc.loc[s.subset, s.feat_type])
    nb = null_bacc[s.model_name]
    na = null_auc[s.model_name]
    nb_m = float(np.mean(nb))
    na_m = float(np.nanmean(na))
    p_b = float((1 + np.sum(np.abs(nb - nb_m) >= abs(ob - nb_m))) / (len(nb) + 1))
    p_a = float((1 + np.sum(np.abs(na - na_m) >= abs(oa - na_m))) / (len(na) + 1))
    perm_rows.append({
        "model": s.model_name, "subset": s.subset, "feat_type": s.feat_type,
        "obs_bacc": ob, "null_bacc_mean": nb_m, "p_perm_bacc": p_b,
        "obs_auc":  oa, "null_auc_mean":  na_m, "p_perm_auc":  p_a,
    })
perm_df = pd.DataFrame(perm_rows).sort_values("p_perm_bacc").reset_index(drop=True)
display(perm_df.round(4))

Running 1000 permutations across 60 workers …


,model,subset,feat_type,obs_bacc,null_bacc_mean,p_perm_bacc,obs_auc,null_auc_mean,p_perm_auc
0,right_lateral__B,right_lateral,B,0.3584,0.5008,0.0020,0.3280,0.5016,0.0120
1,both_lateral__C,both_lateral,C,0.7273,0.5011,0.0020,0.8019,0.5018,0.0070
2,left_lateral__A,left_lateral,A,0.7310,0.4997,0.0060,0.8014,0.4981,0.0070
3,both_lateral__A,both_lateral,A,0.7540,0.4998,0.0070,0.8182,0.4997,0.0070
4,left_lateral__C,left_lateral,C,0.7353,0.4988,0.0110,0.8057,0.4983,0.0030
5,right_lateral__A,right_lateral,A,0.7043,0.4994,0.0130,0.7701,0.4996,0.0260
6,bis_quatro__A,bis_quatro,A,0.6333,0.4977,0.0769,0.7031,0.4967,0.0190
7,right_lateral__C,right_lateral,C,0.6436,0.5009,0.0849,0.7209,0.5014,0.0480
8,bis_quatro__C,bis_quatro,C,0.5995,0.4974,0.1538,0.6723,0.4964,0.0899
9,bis_quatro__B,bis_quatro,B,0.4625,0.4981,0.3836,0.4547,0.4961,0.4386


In [13]:
# %%
# Save permutation outputs
OUT_PERM_DIR = PROJECT_ROOT / "results" / "permutation_results"
OUT_PERM_DIR.mkdir(parents=True, exist_ok=True)

perm_df.to_csv(OUT_PERM_DIR / "perm_summary_channel_subsets.csv", index=False)
np.savez(OUT_PERM_DIR / "null_channel_subsets_bacc.npz", **null_bacc)
np.savez(OUT_PERM_DIR / "null_channel_subsets_auc.npz",  **null_auc)
print("Saved permutation outputs to:", OUT_PERM_DIR)

Saved permutation outputs to: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/permutation_results


In [14]:
# %%
# ============================================
# Section 6. Summary table with full-62ch baselines
# ============================================

full = pd.read_csv(OUT_DIR / "results_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv")

baseline_rows = [
    {"subset": "full_62ch", "feat_type": "A", "n_ch": 62, "n_feat": 26,
     "bacc_mean": full[full["model"]=="A_bandpower_epoch"]["balanced_accuracy"].mean(),
     "bacc_std":  full[full["model"]=="A_bandpower_epoch"]["balanced_accuracy"].std(),
     "auc_mean":  full[full["model"]=="A_bandpower_epoch"]["roc_auc"].mean(),
     "auc_std":   full[full["model"]=="A_bandpower_epoch"]["roc_auc"].std()},
    {"subset": "full_62ch", "feat_type": "B", "n_ch": 62, "n_feat": 5673,
     "bacc_mean": full[full["model"]=="B_wpli_edges_epoch_sliding"]["balanced_accuracy"].mean(),
     "bacc_std":  full[full["model"]=="B_wpli_edges_epoch_sliding"]["balanced_accuracy"].std(),
     "auc_mean":  full[full["model"]=="B_wpli_edges_epoch_sliding"]["roc_auc"].mean(),
     "auc_std":   full[full["model"]=="B_wpli_edges_epoch_sliding"]["roc_auc"].std()},
    {"subset": "full_62ch", "feat_type": "C", "n_ch": 62, "n_feat": 5699,
     "bacc_mean": full[full["model"]=="C_AplusB_epoch_sliding"]["balanced_accuracy"].mean(),
     "bacc_std":  full[full["model"]=="C_AplusB_epoch_sliding"]["balanced_accuracy"].std(),
     "auc_mean":  full[full["model"]=="C_AplusB_epoch_sliding"]["roc_auc"].mean(),
     "auc_std":   full[full["model"]=="C_AplusB_epoch_sliding"]["roc_auc"].std()},
]

n_ch_map   = {"left_lateral": 5, "right_lateral": 5, "both_lateral": 10, "bis_quatro": 4}
n_feat_map = {
    s.model_name: s.n_features for s in all_specs
}

subset_rows = []
for (subset, feat_type), grp in folds_df.groupby(["subset", "feat_type"]):
    model_name = f"{subset}__{feat_type}"
    subset_rows.append({
        "subset":     subset,
        "feat_type":  feat_type,
        "n_ch":       n_ch_map[subset],
        "n_feat":     n_feat_map[model_name],
        "bacc_mean":  grp["balanced_accuracy"].mean(),
        "bacc_std":   grp["balanced_accuracy"].std(),
        "auc_mean":   grp["roc_auc"].mean(),
        "auc_std":    grp["roc_auc"].std(),
    })

summary = pd.DataFrame(baseline_rows + subset_rows).round(4)
summary = summary.sort_values(["feat_type", "bacc_mean"], ascending=[True, False])
print("\n=== Full comparison (sorted by feat_type then balanced accuracy) ===")
display(summary.set_index(["subset", "feat_type"]))


=== Full comparison (sorted by feat_type then balanced accuracy) ===


,,n_ch,n_feat,bacc_mean,bacc_std,auc_mean,auc_std
subset,feat_type,,,,,,
both_lateral,A,10,40,0.7540,0.1424,0.8182,0.1480
left_lateral,A,5,20,0.7310,0.1448,0.8014,0.1344
full_62ch,A,62,26,0.7057,0.1481,0.8214,0.1586
right_lateral,A,5,20,0.7043,0.1333,0.7701,0.1342
bis_quatro,A,4,16,0.6333,0.0844,0.7031,0.1134
full_62ch,B,62,5673,0.5294,0.0792,0.5355,0.0682
left_lateral,B,5,30,0.4979,0.0577,0.5448,0.0538
both_lateral,B,10,135,0.4770,0.0893,0.4380,0.0794
bis_quatro,B,4,18,0.4625,0.0455,0.4547,0.0725


In [15]:
# %%
# ============================================
# Section 7. Plot 1 — Grouped bar chart: balanced accuracy by subset and feature type
# ============================================

subsets_order = ["full_62ch", "both_lateral", "left_lateral", "right_lateral", "bis_quatro"]
feat_colors   = {"A": "steelblue", "B": "coral", "C": "seagreen"}
n_subsets     = len(subsets_order)
n_feat_types  = 3
width         = 0.22
x             = np.arange(n_subsets)

fig, ax = plt.subplots(figsize=(13, 5))

for fi, (feat_type, color) in enumerate(feat_colors.items()):
    means, errs = [], []
    for subset in subsets_order:
        row = summary[(summary["subset"] == subset) & (summary["feat_type"] == feat_type)]
        means.append(float(row["bacc_mean"].values[0]) if len(row) else np.nan)
        errs.append(float(row["bacc_std"].values[0])  if len(row) else 0)
    offset = (fi - 1) * width
    ax.bar(x + offset, means, width, color=color, alpha=0.85,
           yerr=errs, capsize=4, label=f"Feature set {feat_type}",
           error_kw={"linewidth": 1.5})

ax.axhline(0.5, linestyle="--", color="grey", linewidth=1)
labels = ["Full 62ch", "Both lateral\n(10ch)", "Left lateral\n(5ch)", "Right lateral\n(5ch)", "BIS Quatro\n(4ch)"]
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Balanced accuracy (mean ± std, 5 folds)", fontsize=11)
ax.set_title("Channel subset comparison — A (bandpower) / B (wPLI) / C (A+B)\n"
             "GridSearchCV exhaustive | eyes-closed | 10 subjects", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0.4, 1.0)
fig.tight_layout()

out = FIGURE_DIR / "channel_subsets_bacc_bar.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/channel_subsets_bacc_bar.png


In [16]:
# %%
# ============================================
# Section 8. Plot 2 — Per-fold strip plot, feature set C only
# (best overall variant; shows fold variance clearly)
# ============================================

subsets_plot  = ["full_62ch", "both_lateral", "left_lateral", "right_lateral", "bis_quatro"]
subset_labels = ["Full 62ch", "Both lateral (10ch)",
                 "Left lateral (5ch)", "Right lateral (5ch)", "BIS Quatro (4ch)"]
subset_colors = ["dimgrey", "purple", "steelblue", "coral", "seagreen"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, feat_type in zip(axes, ["A", "B", "C"]):
    for i, (subset, label, color) in enumerate(zip(subsets_plot, subset_labels, subset_colors)):
        if subset == "full_62ch":
            model_key = {"A": "A_bandpower_epoch",
                         "B": "B_wpli_edges_epoch_sliding",
                         "C": "C_AplusB_epoch_sliding"}[feat_type]
            vals = full[full["model"] == model_key]["balanced_accuracy"].values
        else:
            vals = folds_df[(folds_df["subset"] == subset) &
                            (folds_df["feat_type"] == feat_type)]["balanced_accuracy"].values
        jitter = (np.random.RandomState(i).rand(len(vals)) - 0.5) * 0.10
        ax.scatter(np.full(len(vals), i) + jitter, vals,
                   color=color, s=55, zorder=3, alpha=0.9)
        ax.hlines(vals.mean(), i - 0.25, i + 0.25, colors=color, linewidths=2.5)

    ax.axhline(0.5, linestyle=":", color="grey", linewidth=1)
    ax.set_xticks(range(len(subsets_plot)))
    ax.set_xticklabels(subset_labels, fontsize=8, rotation=25, ha="right")
    ax.set_title(f"Feature set {feat_type}", fontsize=12)
    ax.set_ylim(0.35, 1.0)

axes[0].set_ylabel("Balanced accuracy", fontsize=11)
fig.suptitle("Per-fold balanced accuracy by channel subset and feature type", fontsize=12, y=1.01)
fig.tight_layout()

out = FIGURE_DIR / "channel_subsets_per_fold.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/channel_subsets_per_fold.png


In [17]:
# %%
# ============================================
# Section 9. Plot 3 — performance heatmap (subset × feature type)
# Stars: two-sided permutation test against shuffled-label null
#   *** p<0.001   ** p<0.01   * p<0.05
# Subset rows:    p-values from perm_df (this notebook, balanced-acc / AUC null).
# Full-62ch row:  p-values from results/permutation_results/perm_summary.csv
#                 (computed in 04_analysis on accuracy null — used for both panels).
# ============================================

feat_types    = ["A", "B", "C"]
subsets_heat  = ["full_62ch", "both_lateral", "left_lateral", "right_lateral", "bis_quatro"]
heat_labels   = ["Full 62ch", "Both lateral (10)", "Left lateral (5)", "Right lateral (5)", "BIS Quatro (4)"]

FULL_MODEL_MAP = {
    "A": "A_bandpower_epoch",
    "B": "B_wpli_edges_epoch_sliding",
    "C": "C_AplusB_epoch_sliding",
}

PERM_FULL_PATH = PROJECT_ROOT / "results" / "permutation_results" / "perm_summary.csv"
full_perm = pd.read_csv(PERM_FULL_PATH).set_index("model") if PERM_FULL_PATH.exists() else None

def p_to_stars(p):
    if pd.isna(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""

auc_mat    = np.full((len(subsets_heat), len(feat_types)), np.nan)
bacc_mat   = np.full((len(subsets_heat), len(feat_types)), np.nan)
p_bacc_mat = np.full((len(subsets_heat), len(feat_types)), np.nan)
p_auc_mat  = np.full((len(subsets_heat), len(feat_types)), np.nan)

for ci, feat_type in enumerate(feat_types):
    for ri, subset in enumerate(subsets_heat):
        row = summary[(summary["subset"] == subset) & (summary["feat_type"] == feat_type)]
        if len(row):
            auc_mat[ri, ci]  = float(row["auc_mean"].values[0])
            bacc_mat[ri, ci] = float(row["bacc_mean"].values[0])
        if subset == "full_62ch":
            if full_perm is not None and FULL_MODEL_MAP[feat_type] in full_perm.index:
                p = float(full_perm.loc[FULL_MODEL_MAP[feat_type], "p_perm_two_sided"])
                p_bacc_mat[ri, ci] = p
                p_auc_mat[ri, ci]  = p
        else:
            prow = perm_df[(perm_df["subset"] == subset) & (perm_df["feat_type"] == feat_type)]
            if len(prow):
                p_bacc_mat[ri, ci] = float(prow["p_perm_bacc"].values[0])
                p_auc_mat[ri, ci]  = float(prow["p_perm_auc"].values[0])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, mat, pmat, title, fmt in [
    (axes[0], bacc_mat, p_bacc_mat, "Balanced accuracy", ".3f"),
    (axes[1], auc_mat,  p_auc_mat,  "ROC-AUC",           ".3f"),
]:
    im = ax.imshow(mat, vmin=0.45, vmax=0.90, cmap="RdYlGn", aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(len(feat_types)))
    ax.set_xticklabels([f"Set {f}" for f in feat_types], fontsize=11)
    ax.set_yticks(range(len(subsets_heat)))
    ax.set_yticklabels(heat_labels, fontsize=10)
    ax.set_title(title, fontsize=12)
    for ri in range(mat.shape[0]):
        for ci in range(mat.shape[1]):
            stars = p_to_stars(pmat[ri, ci])
            label = f"{mat[ri, ci]:{fmt}}"
            if stars:
                label = f"{label}\n{stars}"
            color = "black" if 0.5 < mat[ri, ci] < 0.85 else "white"
            ax.text(ci, ri, label,
                    ha="center", va="center", fontsize=10, color=color)

fig.suptitle("Channel subset × feature type performance heatmap\n"
             "Stars: permutation test vs. shuffled-label null  "
             "(*** p<0.001  ** p<0.01  * p<0.05)",
             fontsize=11, y=1.03)
fig.tight_layout()

out = FIGURE_DIR / "channel_subsets_heatmap.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/channel_subsets_heatmap.png


In [18]:
# %%
print("\n=== Final summary ===")
display(summary.set_index(["subset", "feat_type"]))

print("\nFigures:")
for f in sorted(FIGURE_DIR.glob("channel_subsets_*.png")):
    print(" ", f.name)


=== Final summary ===


,,n_ch,n_feat,bacc_mean,bacc_std,auc_mean,auc_std
subset,feat_type,,,,,,
both_lateral,A,10,40,0.7540,0.1424,0.8182,0.1480
left_lateral,A,5,20,0.7310,0.1448,0.8014,0.1344
full_62ch,A,62,26,0.7057,0.1481,0.8214,0.1586
right_lateral,A,5,20,0.7043,0.1333,0.7701,0.1342
bis_quatro,A,4,16,0.6333,0.0844,0.7031,0.1134
full_62ch,B,62,5673,0.5294,0.0792,0.5355,0.0682
left_lateral,B,5,30,0.4979,0.0577,0.5448,0.0538
both_lateral,B,10,135,0.4770,0.0893,0.4380,0.0794
bis_quatro,B,4,18,0.4625,0.0455,0.4547,0.0725



Figures:
  channel_subsets_bacc_bar.png
  channel_subsets_heatmap.png
  channel_subsets_per_fold.png
